In [1]:
"""
Step 7: Model Evaluation
=========================
"""

import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# 1. LOAD BEST MODEL AND TEST DATA

print("\n1. LOADING BEST MODEL AND TEST DATA")
print("-" * 80)

# Define AttentionLayer for loading (if needed)
class AttentionLayer(keras.layers.Layer):
    """Custom attention layer"""
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
    
    def build(self, input_shape):
        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], input_shape[-1]),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='attention_bias',
            shape=(input_shape[-1],),
            initializer='zeros',
            trainable=True
        )
        super(AttentionLayer, self).build(input_shape)
    
    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = x * a
        return tf.reduce_sum(output, axis=1)
    
    def get_config(self):
        return super(AttentionLayer, self).get_config()

with open('../data/models/best_model_info.pkl', 'rb') as f:
    best_model_info = pickle.load(f)

best_model_name = best_model_info['best_model_name']
print(f"✓ Best Model: {best_model_name}")

# Load the model with custom objects if needed
if best_model_name == 'Attention_LSTM':
    model = keras.models.load_model(
        best_model_info['best_model_path'],
        custom_objects={'AttentionLayer': AttentionLayer}
    )
else:
    model = keras.models.load_model(best_model_info['best_model_path'])
print(f"✓ Loaded model from: {best_model_info['best_model_path']}")

test_data = np.load('../data/processed/test_data.npz')
X_test = test_data['X']
y_test = test_data['y']

print(f"✓ Test set: {X_test.shape[0]:,} samples")

with open('../data/processed/metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

target_names = metadata['target_features']
print(f"✓ Target variables: {target_names}")


1. LOADING BEST MODEL AND TEST DATA
--------------------------------------------------------------------------------
✓ Best Model: CNN_LSTM
✓ Loaded model from: ../data/models/CNN_LSTM_best.keras
✓ Test set: 37,828 samples
✓ Target variables: ['height', 'weight']


In [3]:
# 2. MAKE PREDICTIONS ON TEST SET

print("\n2. GENERATING PREDICTIONS")
print("-" * 80)

y_pred = model.predict(X_test, batch_size=32, verbose=1)

print(f"✓ Predictions shape: {y_pred.shape}")
print(f"✓ Generated {len(y_pred):,} predictions")


2. GENERATING PREDICTIONS
--------------------------------------------------------------------------------
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 1s 635us/step
✓ Predictions shape: (37828, 2)
✓ Generated 37,828 predictions


In [4]:
# 3. CALCULATE PERFORMANCE METRICS

print("\n3. CALCULATING PERFORMANCE METRICS")
print("-" * 80)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nOverall Performance:")
print(f"  MSE: {mse:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  R² Score: {r2:.4f}")

print("\nPer-Target Performance:")
for i, target_name in enumerate(target_names):
    target_mse = mean_squared_error(y_test[:, i], y_pred[:, i])
    target_rmse = np.sqrt(target_mse)
    target_mae = mean_absolute_error(y_test[:, i], y_pred[:, i])
    target_r2 = r2_score(y_test[:, i], y_pred[:, i])
    
    print(f"\n{target_name.upper()}:")
    print(f"  MSE: {target_mse:.4f}")
    print(f"  RMSE: {target_rmse:.4f}")
    print(f"  MAE: {target_mae:.4f}")
    print(f"  R² Score: {target_r2:.4f}")


3. CALCULATING PERFORMANCE METRICS
--------------------------------------------------------------------------------

Overall Performance:
  MSE: 25.9644
  RMSE: 5.0955
  MAE: 2.9742
  R² Score: 0.5253

Per-Target Performance:

HEIGHT:
  MSE: 43.5994
  RMSE: 6.6030
  MAE: 4.4068
  R² Score: 0.7316

WEIGHT:
  MSE: 8.3298
  RMSE: 2.8861
  MAE: 1.5417
  R² Score: 0.3189


In [5]:
# 4. ANALYZE PREDICTION ERRORS

print("\n4. ANALYZING PREDICTION ERRORS")
print("-" * 80)

errors = y_test - y_pred
absolute_errors = np.abs(errors)

print("\nError Statistics:")
for i, target_name in enumerate(target_names):
    print(f"\n{target_name.upper()} Error Distribution:")
    print(f"  Mean Error: {errors[:, i].mean():.4f}")
    print(f"  Std Error: {errors[:, i].std():.4f}")
    print(f"  Mean Absolute Error: {absolute_errors[:, i].mean():.4f}")
    print(f"  Median Absolute Error: {np.median(absolute_errors[:, i]):.4f}")
    print(f"  90th Percentile Error: {np.percentile(absolute_errors[:, i], 90):.4f}")
    print(f"  95th Percentile Error: {np.percentile(absolute_errors[:, i], 95):.4f}")



4. ANALYZING PREDICTION ERRORS
--------------------------------------------------------------------------------

Error Statistics:

HEIGHT Error Distribution:
  Mean Error: 0.1144
  Std Error: 6.6020
  Mean Absolute Error: 4.4068
  Median Absolute Error: 2.7831
  90th Percentile Error: 10.4641
  95th Percentile Error: 14.2685

WEIGHT Error Distribution:
  Mean Error: -0.0594
  Std Error: 2.8855
  Mean Absolute Error: 1.5417
  Median Absolute Error: 1.1696
  90th Percentile Error: 3.1334
  95th Percentile Error: 3.9556


In [6]:
# 5. VISUALIZE PREDICTIONS VS ACTUALS

print("\n5. CREATING VISUALIZATION PLOTS")
print("-" * 80)

fig = plt.figure(figsize=(20, 12))

for i, target_name in enumerate(target_names):
    ax1 = plt.subplot(3, 2, i*2 + 1)
    plt.scatter(y_test[:, i], y_pred[:, i], alpha=0.3, s=10)
    
    min_val = min(y_test[:, i].min(), y_pred[:, i].min())
    max_val = max(y_test[:, i].max(), y_pred[:, i].max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    plt.xlabel(f'Actual {target_name.capitalize()}', fontsize=12)
    plt.ylabel(f'Predicted {target_name.capitalize()}', fontsize=12)
    plt.title(f'{target_name.capitalize()}: Predicted vs Actual\nR² = {r2_score(y_test[:, i], y_pred[:, i]):.4f}', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    ax2 = plt.subplot(3, 2, i*2 + 2)
    plt.hist(errors[:, i], bins=50, edgecolor='black', alpha=0.7)
    plt.axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero Error')
    plt.xlabel(f'Prediction Error ({target_name})', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title(f'{target_name.capitalize()}: Error Distribution\nMAE = {mean_absolute_error(y_test[:, i], y_pred[:, i]):.4f}', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)

ax3 = plt.subplot(3, 2, 5)
for i, target_name in enumerate(target_names):
    plt.scatter(y_pred[:, i], errors[:, i], alpha=0.3, s=10, label=target_name.capitalize())
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Value', fontsize=12)
plt.ylabel('Residual (Actual - Predicted)', fontsize=12)
plt.title('Residual Plot', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

ax4 = plt.subplot(3, 2, 6)
for i, target_name in enumerate(target_names):
    bins = pd.qcut(y_pred[:, i], q=20, duplicates='drop')
    binned_mae = pd.DataFrame({
        'pred': y_pred[:, i],
        'error': absolute_errors[:, i],
        'bin': bins
    }).groupby('bin')['error'].mean()
    
    bin_centers = pd.DataFrame({
        'pred': y_pred[:, i],
        'bin': bins
    }).groupby('bin')['pred'].mean()
    
    plt.plot(bin_centers, binned_mae, marker='o', linewidth=2, label=target_name.capitalize())

plt.xlabel('Predicted Value', fontsize=12)
plt.ylabel('Mean Absolute Error', fontsize=12)
plt.title('Error by Prediction Magnitude', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/models/evaluation_plots.png', dpi=300, bbox_inches='tight')
print("✓ Saved evaluation plots to: ../data/models/evaluation_plots.png")
plt.close()


5. CREATING VISUALIZATION PLOTS
--------------------------------------------------------------------------------
✓ Saved evaluation plots to: ../data/models/evaluation_plots.png


In [7]:
# 6. ERROR ANALYSIS BY PERCENTILE

print("\n6. ERROR ANALYSIS BY PERCENTILE")
print("-" * 80)

percentiles = [50, 75, 90, 95, 99]

print("\nPercentile Analysis of Absolute Errors:")
for i, target_name in enumerate(target_names):
    print(f"\n{target_name.upper()}:")
    for p in percentiles:
        error_at_p = np.percentile(absolute_errors[:, i], p)
        print(f"  {p}th percentile: {error_at_p:.4f}")


6. ERROR ANALYSIS BY PERCENTILE
--------------------------------------------------------------------------------

Percentile Analysis of Absolute Errors:

HEIGHT:
  50th percentile: 2.7831
  75th percentile: 5.6774
  90th percentile: 10.4641
  95th percentile: 14.2685
  99th percentile: 23.6553

WEIGHT:
  50th percentile: 1.1696
  75th percentile: 2.0581
  90th percentile: 3.1334
  95th percentile: 3.9556
  99th percentile: 6.0502


In [8]:
# 7. SAVE EVALUATION RESULTS

print("\n7. SAVING EVALUATION RESULTS")
print("-" * 80)

evaluation_results = {
    'model_name': best_model_name,
    'test_samples': len(X_test),
    'overall_metrics': {
        'mse': float(mse),
        'rmse': float(rmse),
        'mae': float(mae),
        'r2': float(r2)
    },
    'per_target_metrics': {}
}

for i, target_name in enumerate(target_names):
    evaluation_results['per_target_metrics'][target_name] = {
        'mse': float(mean_squared_error(y_test[:, i], y_pred[:, i])),
        'rmse': float(np.sqrt(mean_squared_error(y_test[:, i], y_pred[:, i]))),
        'mae': float(mean_absolute_error(y_test[:, i], y_pred[:, i])),
        'r2': float(r2_score(y_test[:, i], y_pred[:, i])),
        'mean_error': float(errors[:, i].mean()),
        'std_error': float(errors[:, i].std()),
        'median_absolute_error': float(np.median(absolute_errors[:, i])),
        'p90_error': float(np.percentile(absolute_errors[:, i], 90)),
        'p95_error': float(np.percentile(absolute_errors[:, i], 95))
    }

with open('../data/models/evaluation_results.pkl', 'wb') as f:
    pickle.dump(evaluation_results, f)

print("✓ ../data/models/evaluation_results.pkl")

np.savez_compressed('../data/models/test_predictions.npz',
                    y_true=y_test,
                    y_pred=y_pred,
                    errors=errors)

print("✓ Saved test predictions to: ../data/models/test_predictions.npz")



7. SAVING EVALUATION RESULTS
--------------------------------------------------------------------------------
✓ ../data/models/evaluation_results.pkl
✓ Saved test predictions to: ../data/models/test_predictions.npz


In [9]:
# 8. CREATE EVALUATION REPORT

print("\n8. GENERATING EVALUATION REPORT")
print("-" * 80)

report_path = '../data/models/evaluation_report.txt'

with open(report_path, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("CHILD GROWTH PREDICTION MODEL - EVALUATION REPORT\n")
    f.write("=" * 80 + "\n\n")
    
    f.write(f"Model: {best_model_name}\n")
    f.write(f"Test Samples: {len(X_test):,}\n\n")
    
    f.write("OVERALL PERFORMANCE\n")
    f.write("-" * 80 + "\n")
    f.write(f"MSE: {mse:.4f}\n")
    f.write(f"RMSE: {rmse:.4f}\n")
    f.write(f"MAE: {mae:.4f}\n")
    f.write(f"R² Score: {r2:.4f}\n\n")
    
    for i, target_name in enumerate(target_names):
        f.write(f"\n{target_name.upper()} METRICS\n")
        f.write("-" * 80 + "\n")
        metrics = evaluation_results['per_target_metrics'][target_name]
        for metric, value in metrics.items():
            f.write(f"{metric}: {value:.4f}\n")
    
    f.write("\n" + "=" * 80 + "\n")
    f.write("INTERPRETATION\n")
    f.write("=" * 80 + "\n\n")
    
    for i, target_name in enumerate(target_names):
        mae_val = evaluation_results['per_target_metrics'][target_name]['mae']
        p90_val = evaluation_results['per_target_metrics'][target_name]['p90_error']
        p95_val = evaluation_results['per_target_metrics'][target_name]['p95_error']
        
        f.write(f"{target_name.upper()} Predictions:\n")
        f.write(f"- Average error: ±{mae_val:.2f} units\n")
        f.write(f"- 90% of predictions within: ±{p90_val:.2f} units\n")
        f.write(f"- 95% of predictions within: ±{p95_val:.2f} units\n\n")

print(f"✓ Saved evaluation report to: {report_path}")


8. GENERATING EVALUATION REPORT
--------------------------------------------------------------------------------
✓ Saved evaluation report to: ../data/models/evaluation_report.txt


In [10]:
# 9. EVALUATION SUMMARY

print("\n" + "=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)

print(f"\n✓ Model: {best_model_name}")
print(f"✓ Test samples evaluated: {len(X_test):,}")

print("\nFinal Performance Metrics:")
print(f"  Overall MAE: {mae:.4f}")
print(f"  Overall RMSE: {rmse:.4f}")
print(f"  Overall R²: {r2:.4f}")

print("\nPer-Variable Performance:")
for target_name in target_names:
    metrics = evaluation_results['per_target_metrics'][target_name]
    print(f"\n{target_name.capitalize()}:")
    print(f"  MAE: ±{metrics['mae']:.2f} (average error)")
    print(f"  90% within: ±{metrics['p90_error']:.2f}")
    print(f"  R²: {metrics['r2']:.4f} (variance explained)")


EVALUATION SUMMARY

✓ Model: CNN_LSTM
✓ Test samples evaluated: 37,828

Final Performance Metrics:
  Overall MAE: 2.9742
  Overall RMSE: 5.0955
  Overall R²: 0.5253

Per-Variable Performance:

Height:
  MAE: ±4.41 (average error)
  90% within: ±10.46
  R²: 0.7316 (variance explained)

Weight:
  MAE: ±1.54 (average error)
  90% within: ±3.13
  R²: 0.3189 (variance explained)
